In [25]:
import pandas as pd
import re

# URL to the raw CSV file on GitHub
url = 'https://raw.githubusercontent.com/LCM-S25-3035/Chatter/refs/heads/dev/1.data/raw/nicolas/ocr_results.csv'

# Read the CSV into a DataFrame
df = pd.read_csv(url)

# Function to perform the fine‐grained cleaning rules
def clean_ocr_text(text):
    cleaned = text.replace('\n', ' ')  # replace newlines with spaces
    cleaned = re.sub(r'[•]{3,}', '', cleaned)  # remove repeated bullets
    cleaned = re.sub(r'(Wikipedia|Artículo|Editar|Ver historial|Desde Wikipedia)[^\n]*', '', cleaned)  # strip headers/footers
    cleaned = cleaned.replace('etc.', 'etcétera')  # expand “etc.”
    cleaned = re.sub(r'\s+', ' ', cleaned)  # collapse multiple spaces
    cleaned = re.sub(r'\s([,\.])', r'\1', cleaned)  # no space before punctuation
    return cleaned.strip()

# Function that cleans text but reverts if >10% is removed
def clean_or_revert(text):
    original_len = len(text)
    cleaned = clean_ocr_text(text)
    if len(cleaned) < 0.9 * original_len:
        return text
    return cleaned

# Apply cleaning to all rows
df['OCR_Text_Cleaned'] = df['OCR_Text'].apply(clean_or_revert)

# Compute length after cleaning
df['Length_After'] = df['OCR_Text_Cleaned'].apply(len)

# Drop the original OCR_Text column
df = df.drop(columns=['OCR_Text'])

# Save to Parquet
output_path = r'C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\Nicolascode\ocr_results_cleaned.parquet'
df.to_parquet(output_path, index=False)

print(f"Cleaned data saved to {output_path}")



Cleaned data saved to C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\Nicolascode\ocr_results_cleaned.parquet


In [16]:
# Extract unique values from the Detected_Language column
unique_languages = df['Detected_Language'].unique()

# Print the result
print("Unique detected languages:", unique_languages)


Unique detected languages: ['en' 'af' 'sl' 'pt' 'nl' 'de' 'id' 'fr' 'pl' 'es' 'lt' 'et' 'ro' 'ca'
 'da' 'no' 'sw' 'it']


In [22]:
# Calculate percentage distribution of detected languages
language_percentages = df['Detected_Language'].value_counts(normalize=True) * 100

# Display the percentages
print("Detected language percentages:")
print(language_percentages)


Detected language percentages:
en    98.439560
id     0.384615
af     0.241758
de     0.208791
es     0.153846
pt     0.153846
pl     0.087912
nl     0.054945
fr     0.054945
lt     0.054945
sl     0.032967
et     0.032967
ca     0.021978
no     0.021978
it     0.021978
ro     0.010989
da     0.010989
sw     0.010989
Name: Detected_Language, dtype: float64


In [21]:
# Select rows where the detected language is 'af' (Afrikaans)
af_rows = df[df['Detected_Language'] == 'fr']

# Display all columns of those rows
print(af_rows)


      Label                                    Path Format            Size  \
1375      0  1c4314aa93e15f005d17ec44b89e741e_3.jpg   JPEG  1224 x 1584 px   
1376      0  1c4314aa93e15f005d17ec44b89e741e_4.jpg   JPEG  1224 x 1584 px   
1377      0  1c4314aa93e15f005d17ec44b89e741e_5.jpg   JPEG  1224 x 1584 px   
3237      0  538a442fc1845445653db3acecc7d534_8.jpg   JPEG  1224 x 1584 px   
8493      1  c6cdcbc46deb96c0d08f361b0b4a5860_4.jpg   JPEG  1224 x 1584 px   

                                               OCR_Text  \
1375  . Madeleine | TV series (1\nSpiral . Pascal Ch...   
1376  Jér6me Commandeur & Alan\n\nMa famille t'adore...   
1377  2002- | Un petit jeu sans Jean Dell & Gérald ....   
3237  == Belisario Agulla ¢ Ef Christian Delage\n\n=...   
8493  e Chevalier des Grieux, Manon Lescaut (Puccini...   

                                       OCR_Text_Cleaned  Length_Before  \
1375  . Madeleine | TV series (1 Spiral. Pascal Chau...           1394   
1376  Jér6me Commandeur & Al

In [23]:
# Save the resulting DataFrame to a new CSV
output_path = r'C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\Nicolascode\ocr_results_cleaned.csv'
df.to_csv(output_path, index=False)

In [24]:
df.head()

,Label,Path,Format,Size,OCR_Text,OCR_Text_Cleaned,Length_Before,Length_After,Detected_Language
0,0,00163fe7688e71ce06f495a6811fef71_1.jpg,JPEG,1224 x 1584 px,WIKIPEDIA\n\n: Donate Create account Log in\n‘...,WIKIPEDIA\n\n: Donate Create account Log in\n‘...,2283,2283,en
1,0,00163fe7688e71ce06f495a6811fef71_2.jpg,JPEG,1224 x 1584 px,Animation {ecit]\n\nYear Title Role Notes Sour...,Animation {ecit] Year Title Role Notes Source ...,1354,1344,en
2,0,00163fe7688e71ce06f495a6811fef71_3.jpg,JPEG,1224 x 1584 px,Pretty Pretty Please | Don't | Sidney the . .\...,Pretty Pretty Please | Don't | Sidney the........,1227,1215,en
3,0,00163fe7688e71ce06f495a6811fef71_4.jpg,JPEG,1224 x 1584 px,"Brooke Page, Darling\n\n2015 | Ever After High...","Brooke Page, Darling 2015 | Ever After High: W...",1208,1201,en
4,0,00163fe7688e71ce06f495a6811fef71_5.jpg,JPEG,1224 x 1584 px,Live-action [edit]\nYear Title Role Notes Sour...,Live-action [edit] Year Title Role Notes Sourc...,1363,1341,en
